# 04 — Backward Op（反向传播算子）

## 背景

深度学习框架在反向传播时自动生成梯度算子。然而对于自定义算子或追求极致性能的场景，手写 CUDA/Triton/TileLang 梯度内核是必要的。

本节以**广播乘法 + ReLU** 为例，展示如何同时实现前向和反向算子，并对比三种框架的写法与性能。

## 问题定义

**前向传播（Broadcast Mul + ReLU）**：
$$C[i, j] = \max\bigl(0,\ A[i, j] \times B[j]\bigr)$$

**反向传播（链式法则，对 A 的梯度）**：
$$\frac{\partial\mathcal{L}}{\partial A[i,j]} = \frac{\partial\mathcal{L}}{\partial C[i,j]} \times B[j] \times \mathbf{1}[A[i,j] \times B[j] > 0]$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N, M = 8192, 4096
A  = torch.randn(N, M, dtype=torch.float16, device="cuda")
B  = torch.randn(M,    dtype=torch.float16, device="cuda")
dC = torch.randn(N, M, dtype=torch.float16, device="cuda")

def ref_fwd(A, B):
    return torch.relu(A * B)

def ref_bwd(A, B, dC):
    # Direct gradient: dA = dC * B * 1[A*B > 0], computed in fp16.
    # In-place .mul_() avoids materialising an explicit float-cast tensor:
    #   current (dC*B*(A*B>0).to(fp16)) = 5 kernel launches on iGPU
    #   in-place version                = 3 kernel launches → ~20% faster on iGPU
    return dC.mul(B).mul_(A.mul(B).gt(0))

C_ref  = ref_fwd(A, B)
dA_ref = ref_bwd(A, B, dC)
print(f"Forward:  {A.shape} × {B.shape} → {C_ref.shape}")
print(f"Backward: {dC.shape} → dA {dA_ref.shape}")

## Triton 实现

前向：二维网格，每个 Block 处理 `[BN, BM]` 子块，`B[j]` 沿行广播。  
反向：同样结构，增加 ReLU 门控 `(A*B > 0).to(dtype)` 实现次梯度。

In [ ]:
@triton.jit
def _fwd_kernel(A_ptr, B_ptr, C_ptr, N, M, BN: tl.constexpr, BM: tl.constexpr):
    pid_n = tl.program_id(0);  pid_m = tl.program_id(1)
    rows = pid_n * BN + tl.arange(0, BN)
    cols = pid_m * BM + tl.arange(0, BM)
    mask = (rows[:, None] < N) & (cols[None, :] < M)
    a = tl.load(A_ptr + rows[:, None] * M + cols[None, :], mask=mask, other=0.0)
    b = tl.load(B_ptr + cols, mask=cols < M, other=0.0)
    tl.store(C_ptr + rows[:, None] * M + cols[None, :], tl.maximum(a * b[None, :], 0.0), mask=mask)

@triton.jit
def _bwd_kernel(A_ptr, B_ptr, dC_ptr, dA_ptr, N, M, BN: tl.constexpr, BM: tl.constexpr):
    pid_n = tl.program_id(0);  pid_m = tl.program_id(1)
    rows = pid_n * BN + tl.arange(0, BN)
    cols = pid_m * BM + tl.arange(0, BM)
    mask = (rows[:, None] < N) & (cols[None, :] < M)
    a  = tl.load(A_ptr  + rows[:, None] * M + cols[None, :], mask=mask, other=0.0)
    b  = tl.load(B_ptr  + cols, mask=cols < M, other=0.0)
    dc = tl.load(dC_ptr + rows[:, None] * M + cols[None, :], mask=mask, other=0.0)
    gate = (a * b[None, :] > 0).to(a.dtype)
    tl.store(dA_ptr + rows[:, None] * M + cols[None, :], dc * b[None, :] * gate, mask=mask)

# ── GPU-adaptive Triton block shape ──────────────────────────────────────────
# gfx1100 / gfx1201: BN=64, BM=64 — balanced 2D tile, fast discrete memory
# gfx1151 (iGPU, unified memory): BN=1, BM=2048 — one row per program.
#   BN=64 causes stride-M (8KB) writes → cache miss on slow unified memory.
#   BN=1, BM=2048: each program writes 2048 consecutive elements of 1 row → coalesced.
arch_tri = torch.cuda.get_device_properties(0).gcnArchName
if arch_tri.startswith("gfx1151"):
    BN_t, BM_t = 1, 2048
else:
    BN_t, BM_t = 64, 64

def triton_fwd(A, B):
    C = torch.empty_like(A)
    grid = (triton.cdiv(N, BN_t), triton.cdiv(M, BM_t))
    _fwd_kernel[grid](A, B, C, N, M, BN=BN_t, BM=BM_t)
    return C

def triton_bwd(A, B, dC):
    dA = torch.empty_like(A)
    grid = (triton.cdiv(N, BN_t), triton.cdiv(M, BM_t))
    _bwd_kernel[grid](A, B, dC, dA, N, M, BN=BN_t, BM=BM_t)
    return dA

ok_fwd = torch.allclose(C_ref,  triton_fwd(A, B),       atol=1e-2)
dA_ref_f16 = dA_ref
ok_bwd = torch.allclose(dA_ref_f16, triton_bwd(A, B, dC), atol=1e-2)
print("Triton fwd:", "✅" if ok_fwd else "❌", " bwd:", "✅" if ok_bwd else "❌")

## TileLang 实现

In [ ]:
# ── GPU-adaptive config ──────────────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): BN=BM=128, threads=256, shared B
#   Key: cache B in shared mem → 1 read per block. Fwd +68%, bwd +432% vs PyTorch.
# gfx1201 (R9700): same — BN=BM=128, shared B
# gfx1151 (Radeon 8060S): RDNA3.5 iGPU, unified memory.
#   Same strided-write problem as 03_outer: T.Parallel(BN, BM) with large BN
#   causes stride-M writes (8KB stride) → cache miss. Fix: BN=1, BM=2048.
#   Shared B not needed: B (8KB) fits in L2 cache after first access.
#   Result: fwd +99% vs PyTorch, bwd +259% vs PyTorch on gfx1151.
arch = torch.cuda.get_device_properties(0).gcnArchName

if arch.startswith("gfx1151"):
    # gfx1151: BN=1, BM=2048 — coalesced row writes, B served from L2 cache
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_fwd(A, B, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N_, M_), dtype);  B: T.Tensor((M_,), dtype)
        C = T.empty((N_, M_), dtype)
        with T.Kernel(N_, M_ // BLOCK_M, threads=256) as (row, pid_m):
            for j in T.Parallel(BLOCK_M):
                col = pid_m * BLOCK_M + j
                val = A[row, col] * B[col]
                C[row, col] = T.if_then_else(val > dtype(0), val, dtype(0))
        return C

    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_bwd(A, B, dC, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N_, M_), dtype);  B: T.Tensor((M_,), dtype)
        dC: T.Tensor((N_, M_), dtype)
        dA = T.empty((N_, M_), dtype)
        with T.Kernel(N_, M_ // BLOCK_M, threads=256) as (row, pid_m):
            for j in T.Parallel(BLOCK_M):
                col = pid_m * BLOCK_M + j
                gate = T.if_then_else(A[row, col] * B[col] > dtype(0), dtype(1), dtype(0))
                dA[row, col] = dC[row, col] * B[col] * gate
        return dA

    k_fwd = tl_fwd.compile(N=N, M=M, BLOCK_M=2048)
    k_bwd = tl_bwd.compile(N=N, M=M, BLOCK_M=2048)

else:
    # gfx1100 / gfx1201: shared B reduces B reads from N to 1 per block
    BN, BM = 128, 128

    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_fwd(A, B, BLOCK_N: int, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N_, M_), dtype);  B: T.Tensor((M_,), dtype)
        C = T.empty((N_, M_), dtype)
        with T.Kernel(N_ // BLOCK_N, M_ // BLOCK_M, threads=256) as (pn, pm):
            B_s = T.alloc_shared((BLOCK_M,), dtype)
            T.copy(B[pm * BLOCK_M:(pm + 1) * BLOCK_M], B_s)
            for i, j in T.Parallel(BLOCK_N, BLOCK_M):
                r, c = pn * BLOCK_N + i, pm * BLOCK_M + j
                val = A[r, c] * B_s[j]
                C[r, c] = T.if_then_else(val > dtype(0), val, dtype(0))
        return C

    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_bwd(A, B, dC, BLOCK_N: int, BLOCK_M: int):
        N_, M_ = T.const("N, M")
        dtype = T.float16
        A:  T.Tensor((N_, M_), dtype);  B: T.Tensor((M_,), dtype)
        dC: T.Tensor((N_, M_), dtype)
        dA = T.empty((N_, M_), dtype)
        with T.Kernel(N_ // BLOCK_N, M_ // BLOCK_M, threads=256) as (pn, pm):
            B_s = T.alloc_shared((BLOCK_M,), dtype)
            T.copy(B[pm * BLOCK_M:(pm + 1) * BLOCK_M], B_s)
            for i, j in T.Parallel(BLOCK_N, BLOCK_M):
                r, c = pn * BLOCK_N + i, pm * BLOCK_M + j
                gate = T.if_then_else(A[r, c] * B_s[j] > dtype(0), dtype(1), dtype(0))
                dA[r, c] = dC[r, c] * B_s[j] * gate
        return dA

    hyp = {"N": N, "M": M, "BLOCK_N": BN, "BLOCK_M": BM}
    k_fwd = tl_fwd.compile(**hyp)
    k_bwd = tl_bwd.compile(**hyp)

ok_f = torch.allclose(C_ref,      k_fwd(A.clone(), B.clone()),             atol=1e-2)
ok_b = torch.allclose(dA_ref_f16, k_bwd(A.clone(), B.clone(), dC.clone()), atol=1e-2)
print("TileLang fwd:", "✅" if ok_f else "❌", " bwd:", "✅" if ok_b else "❌")

## 性能对比

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

fwd_ms = [bench(ref_fwd,    [A, B]),
          bench(triton_fwd, [A, B]),
          bench(k_fwd,      [A, B])]
bwd_ms = [bench(ref_bwd,    [A, B, dC]),
          bench(triton_bwd, [A, B, dC]),
          bench(k_bwd,      [A, B, dC])]

labels = ["PyTorch", "Triton", "TileLang"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

print("Forward: ", {l: f"{ms:.4f} ms" for l, ms in zip(labels, fwd_ms)})
print("Backward:", {l: f"{ms:.4f} ms" for l, ms in zip(labels, bwd_ms)})

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
for ax, data, title in zip(axes, [fwd_ms, bwd_ms],
    [f"Forward (Broadcast Mul+ReLU, {arch})", f"Backward (gradient of A, {arch})"]):
    bars = ax.bar(labels, data, color=colors, width=0.5, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(data)*0.01,
                f"{val:.4f} ms", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"{title}\n(N={N}, M={M}, float16)")
    ax.set_ylim(0, max(data)*1.3)
    ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()